# 8onzas unified agent — menu evals (lab A1)

Canonical module: `lab/agent_v1.py` — **two-phase** (Groq-safe): ReAct tools, then structured `AgentResponse` without tools.

This notebook focuses on **menu/hours** precision/recall. Same `build_agent` as availability notebook.

Requires `GROQ_API_KEY` in `.env` (fallback: `OPENROUTER_API_KEY`). Do not commit secrets.

In [1]:
import os
import sys
from pathlib import Path

# Allow importing lab helpers when the kernel cwd is repo root or lab/
LAB_DIR = Path.cwd()
if LAB_DIR.name != "lab":
    candidate = LAB_DIR / "lab"
    if candidate.exists():
        sys.path.insert(0, str(candidate))
else:
    sys.path.insert(0, str(LAB_DIR))

from agent_v1 import (
    AgentResponse,
    build_agent,
    build_system_prompt,
    evaluate,
    invoke_agent,
    load_calendar,
    load_cases,
    load_dotenv,
    load_menu,
    load_profile,
    resolve_groq_api_key,
)

load_dotenv()
resolve_groq_api_key()
profile = load_profile()
menu = load_menu()
calendar = load_calendar()
suite = load_cases()
profile.name, len(menu.categories), len(suite.cases)

('8onzas', 3, 4)

## System prompt preview

In [2]:
prompt = build_system_prompt(profile, menu, calendar)
print(prompt[:800], "\n...")

Sos el asistente de 8onzas del café 8onzas (Argentina).

Capacidades:
- Menú: platos, precios, tags (ej. vegano). Precios SOLO del JSON de menú (ARS).
- Horario: apertura/cierre por día del perfil.
- Disponibilidad de mesas: SOLO vía la tool check_availability (no inventes slots).
- Si preguntan por un producto que NO está en el menú: product.found=false.

Reglas:
- Contestá en español, tono claro y hospitalario.
- Ids de producto exactos del menú (ej. latte, bowl-vegano).
- La ÚNICA tool disponible es check_availability. No inventes otras tools.
- Para disponibilidad / reserva / mesa para N en fecha+hora: llamá check_availability
  cuando haga falta; si la tool devuelve error, corregí los args y reintentá.
- Después de usar tools (si aplica), respondé en texto claro al usuario con los dat 
...


## Build `create_agent` + smoke invoke

In [3]:
agent = build_agent(profile, menu, calendar)
smoke = invoke_agent("¿Cuánto sale el latte?", agent=agent)
assert isinstance(smoke, AgentResponse)
smoke.model_dump()

{'message': 'El latte tiene un precio de $3\u202f800\u202fARS.',
 'intent': 'menu',
 'product': {'found': True,
  'id': 'latte',
  'name': 'latte',
  'price': 3800.0,
  'currency': 'ARS'},
 'hours': None,
 'availability': None}

## Evaluate suite (precision / recall)

In [4]:
metrics = evaluate(suite, agent=agent)
print(f"precision={metrics['precision']:.3f}  recall={metrics['recall']:.3f}")
print(f"tp={metrics['tp']} fp={metrics['fp']} fn={metrics['fn']}")
print(f"unknown_accuracy={metrics['unknown_accuracy']}  hours_accuracy={metrics['hours_accuracy']}")
metrics["rows"]

precision=1.000  recall=1.000
tp=2 fp=0 fn=0
unknown_accuracy=1.0  hours_accuracy=1.0


[{'id': 'price-latte',
  'capability': 'menu',
  'expected_product_id': 'latte',
  'predicted_product_id': 'latte',
  'product_ok': True,
  'mention_ok': False,
  'intent': 'menu',
  'message': 'El latte tiene un precio de $3\u202f800\u202fARS.',
  'error': None},
 {'id': 'has-vegan-bowl',
  'capability': 'menu',
  'expected_product_id': 'bowl-vegano',
  'predicted_product_id': 'bowl-vegano',
  'product_ok': True,
  'mention_ok': True,
  'intent': 'menu',
  'message': '¡Claro! En nuestro menú contamos con una opción vegana:\n\n| Producto | Descripción | Precio (ARS) |\n|----------|-------------|--------------|\n| **Bowl vegano** (id: `bowl-vegano`) | Quinoa, garbanzos y verduras de estación | **$8\u202f500** |\n\nSi te interesa saber más detalles, precios de otros platos o consultar disponibilidad de mesas para disfrutarlo, solo avísame. ¡Estamos a tu disposición!',
  'error': None},
 {'id': 'hours-friday',
  'capability': 'hours',
  'expected_product_id': None,
  'predicted_product_id

## Notes

- Canonical module: `lab/agent_v1.py` — one agent for menu, hours, and availability.
- Pattern: **Phase 1** ReAct (`check_availability` only) → **Phase 2** `with_structured_output(AgentResponse)` (no tools). Groq forbids combining both in one request.
- Bad tool args → ToolMessage → model can correct in phase 1; phase 2 parse failure → one repair retry.
- This notebook = menu/hours evals; `02_availability_v1.ipynb` = availability evals.
- Promote into `agent/` package after lab metrics look good (`docs/PLAN.md`).